# TP2 — Mini-projet IA (Cas A : Churn clients)

Matière : **Les fondamentaux de l'IA**  
Cas choisi : **A — Churn clients (IBM Telco Customer Churn)**

Ce notebook suit strictement les **6 étapes** demandées dans `TP2.pdf`.


In [ ]:
# Installation (si nécessaire) + imports
import sys
import subprocess
from pathlib import Path

required = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "xgboost",
    "shap",
    "requests",
]

try:
    import pandas
    import numpy
    import sklearn
    import xgboost
    import shap
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *required])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import shap

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

ASSETS_DIR = Path("../assets")
ASSETS_DIR.mkdir(parents=True, exist_ok=True)


## Étape 1 — Cadrage du projet

### Tableau de cadrage (Cas A : Churn)

| Dimension | Votre réponse |
|---|---|
| Problème métier | Identifier les clients à risque de résiliation pour déclencher des actions de rétention ciblées. |
| KPI métier | Taux de churn mensuel, taux de rétention après campagne, revenu conservé (MRR sauvegardé). |
| Variable cible (y) | `Churn` (No/Yes), encodée ensuite en binaire 0/1. |
| Type de tâche ML | Classification binaire. |
| Métrique ML principale | **F1-macro** (dataset déséquilibré, on veut équilibrer performance entre classes churn/non-churn). |
| Risques identifiés | Biais possibles sur profils de clients (senior, type de contrat, mode de paiement). Risque de sur-solliciter certains segments. |
| Niveau AI Act | **Risque limité** (support à la décision commerciale, pas une décision automatisée à fort impact juridique). |

### Réponses aux questions guidées

- **Erreur la plus coûteuse** : le **faux négatif** (client réellement à risque mais non détecté), car on perd un client sans action préventive.
- **Populations potentiellement discriminées** : seniors, clients à faibles revenus (proxy via options/contrat), clients digitalement moins actifs. Un audit de biais par segment est nécessaire.


## Étape 2 — Préparation des données

Objectif de fin d'étape : définir `X_train_sc`, `X_test_sc`, `y_train`, `y_test`, `feature_names`.

Prétraitements appliqués :
1. Conversion de `TotalCharges` en numérique (gestion des espaces/valeurs invalides).
2. Suppression des lignes avec valeurs manquantes.
3. Encodage des variables catégorielles par `get_dummies`.
4. Split stratifié train/test.
5. Normalisation des variables explicatives.


In [ ]:
# Chargement depuis URL publique — IBM Telco Customer Churn
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print(f"Forme initiale : {df.shape}")
print("\nDistribution cible :")
print(df["Churn"].value_counts())

# Nettoyage : TotalCharges contient des espaces
missing_before = df["TotalCharges"].isna().sum()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
missing_after = df["TotalCharges"].isna().sum()

df.dropna(inplace=True)
print(f"\nTotalCharges NA avant conversion : {missing_before}")
print(f"TotalCharges NA après conversion : {missing_after}")
print(f"Après nettoyage : {df.shape}")

# Encodage des catégorielles
# customerID est un identifiant technique (non prédictif), on le retire
df_enc = pd.get_dummies(df.drop("customerID", axis=1), drop_first=True)

X = df_enc.drop("Churn_Yes", axis=1)
y = df_enc["Churn_Yes"].astype(int)
feature_names = list(X.columns)

print(f"\nTaux de churn : {y.mean() * 100:.2f}%")
print("=> Dataset déséquilibré : F1-macro prioritaire.")

# Split stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisation
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"\nTrain : {len(X_train)} | Test : {len(X_test)}")
print(f"Nombre de features : {len(feature_names)}")


### Questions Étape 2 — Réponses

1. **Dataset équilibré ?**  
   Non. Le churn est autour de **26,6%** (classe minoritaire), non-churn autour de **73,4%**.

2. **Variables potentiellement sensibles ?**  
   Oui, par exemple `SeniorCitizen` peut être sensible. Il ne faut pas exclure automatiquement : il faut auditer l'impact (équité, performance par sous-groupes) et documenter l'usage.

3. **Prétraitement appliqué et pourquoi ?**  
   Conversion `TotalCharges`, suppression NA, encodage one-hot des catégorielles, split stratifié pour préserver la proportion de churn, normalisation pour stabiliser l'entraînement (notamment Logistic Regression).


## Étape 3 — Modélisation : comparaison de 3 modèles

Modèles imposés :
- Logistic Regression
- Random Forest
- XGBoost


In [ ]:
modeles = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="logloss",
        verbosity=0,
    ),
}

resultats = {}
predictions = {}

for nom, modele in modeles.items():
    modele.fit(X_train_sc, y_train)
    pred = modele.predict(X_test_sc)
    predictions[nom] = pred

    acc = accuracy_score(y_test, pred)
    f1_mac = f1_score(y_test, pred, average="macro")
    f1_wei = f1_score(y_test, pred, average="weighted")

    resultats[nom] = {
        "accuracy": acc,
        "f1_macro": f1_mac,
        "f1_weighted": f1_wei,
    }

    print(f"{nom:22s} → Accuracy : {acc*100:.2f}% | F1-macro : {f1_mac:.3f} | F1-weighted : {f1_wei:.3f}")

df_resultats = pd.DataFrame(resultats).T.sort_values("f1_macro", ascending=False)
print("\n=== TABLEAU COMPARATIF ===")
display(df_resultats.round(3))

ax = df_resultats[["accuracy", "f1_macro", "f1_weighted"]].plot(kind="bar", figsize=(10, 4))
ax.set_title("Comparaison des modèles (Cas A — Churn)")
ax.set_ylabel("Score")
ax.set_xlabel("Modèle")
ax.set_ylim(0.5, 1.0)
ax.grid(axis="y", alpha=0.4)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(ASSETS_DIR / "comparaison_modeles.png", bbox_inches="tight")
plt.show()


### Questions Étape 3 — Réponses

1. **Quel modèle obtient les meilleures performances ?**  
   Sur ce split, **Logistic Regression** est la meilleure en F1-macro.

2. **La régression logistique est-elle compétitive ?**  
   Oui, elle est même la meilleure ici, ce qui confirme qu'un modèle simple peut très bien performer avec un bon prétraitement.

3. **Modèle recommandé en production ?**  
   **Logistic Regression** pour ce cas : bon compromis performance/interprétabilité/coût de maintenance, temps d'inférence faible.


## Étape 4 — Évaluation approfondie

- Matrice de confusion du meilleur modèle.
- Évolution du F1-macro de Random Forest en fonction de `n_estimators`.


In [ ]:
# Choisir le meilleur modèle selon F1-macro
NOM_MEILLEUR = df_resultats.index[0]
meilleur = modeles[NOM_MEILLEUR]
y_pred_best = meilleur.predict(X_test_sc)

print(f"=== RAPPORT DÉTAILLÉ — {NOM_MEILLEUR} ===")
print(classification_report(y_test, y_pred_best))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Matrice de confusion — {NOM_MEILLEUR}")
plt.ylabel("Réalité")
plt.xlabel("Prédiction")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "confusion_matrix_best_model.png", bbox_inches="tight")
plt.show()

# Évolution de Random Forest selon le nombre d'arbres
n_estimators_range = [10, 25, 50, 100, 200]
scores_rf = []

for n in n_estimators_range:
    rf_temp = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_temp.fit(X_train_sc, y_train)
    pred_temp = rf_temp.predict(X_test_sc)
    scores_rf.append(f1_score(y_test, pred_temp, average="macro"))

plt.figure(figsize=(8, 4))
plt.plot(n_estimators_range, scores_rf, "g-o")
plt.xlabel("Nombre d'arbres (n_estimators)")
plt.ylabel("F1-score macro")
plt.title("Évolution des performances — Random Forest")
plt.grid(True)
plt.tight_layout()
plt.savefig(ASSETS_DIR / "rf_f1_vs_n_estimators.png", bbox_inches="tight")
plt.show()


### Analyse critique obligatoire — Étape 4

1. **Faux positifs vs faux négatifs** : la matrice de confusion met en évidence les deux types d'erreurs; dans le churn, les **faux négatifs** (clients churn non détectés) sont généralement les plus coûteux.

2. **Erreur la plus coûteuse dans ce contexte** : un faux négatif implique une perte de revenu et un coût d'acquisition futur plus élevé pour compenser le client perdu.

3. **Suffisant pour production ?**  
   Pas seul. Il faut ajouter : calibration des seuils, suivi drift/data quality, monitoring des performances par segment, et validation métier continue.


## Étape 5 — Explicabilité SHAP

Analyse SHAP réalisée avec **Random Forest** (TreeExplainer), comme demandé dans le TP.


In [ ]:
# SHAP avec Random Forest
rf_model = modeles["Random Forest"]

X_sample = np.array(X_test_sc[:200])
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_sample)

# Compatibilité SHAP
if isinstance(shap_values, list):
    sv = shap_values[1]
elif len(shap_values.shape) == 3:
    sv = shap_values[:, :, 1]
else:
    sv = shap_values

# Summary plot
plt.figure()
shap.summary_plot(
    sv,
    X_sample,
    feature_names=feature_names,
    max_display=15,
    show=False,
)
plt.title("SHAP — Top variables influentes (classe Churn=1)")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "shap_summary.png", bbox_inches="tight")
plt.show()

# Top-3 variables
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_importance = pd.DataFrame(
    {"feature": feature_names, "mean_abs_shap": mean_abs_shap}
).sort_values("mean_abs_shap", ascending=False)

top3 = shap_importance.head(3).reset_index(drop=True)
print("Top 3 variables SHAP :")
display(top3)


### Questions Étape 5 — Réponses

1. **Top-3 variables influentes (SHAP)** : `tenure`, `InternetService_Fiber optic`, `Contract_Two year`.

2. **Cohérence métier** : oui. Une ancienneté élevée (`tenure`) réduit souvent le churn, et le type de service/contrat influence fortement la probabilité de résiliation.

3. **Si une variable sensible apparaît** : lancer une analyse de biais, tester des alternatives (exclusion/contraintes d'équité), documenter la décision et renforcer la supervision humaine.


## Étape 6 — Conformité (AI Act + RGPD)

### Tableau de conformité

| Critère | Votre analyse |
|---|---|
| Niveau de risque AI Act | **Risque limité** |
| Justification du niveau | Cas de marketing/rétention client. Le modèle assiste la décision commerciale, sans automatiser seul une décision juridique équivalente. |
| Base légale RGPD | Principalement **intérêt légitime** (rétention), parfois **contrat** selon le traitement. À documenter dans le registre des traitements. |
| DPIA requis ? | **Souvent non obligatoire** pour ce cas standard, mais recommandé si profilage massif, données sensibles, ou impacts significatifs sur les personnes. |
| Explicabilité requise | Oui, au minimum pour justifier les actions de ciblage (SHAP + documentation modèle). |
| Supervision humaine | Oui, validation humaine des campagnes et des règles métier avant action client. |
| Audit et traçabilité | Versioning datasets/modèles, journalisation des scores, suivi des dérives, gouvernance MLOps (MLflow ou équivalent). |
| Droits des personnes | Droit d'accès, rectification, opposition au profilage/marketing direct, information transparente sur la logique de traitement. |

### Questions Étape 6 — Réponses

1. **Audit avant déploiement ?**  
   Oui, un audit interne conformité + sécurité + data governance est recommandé (DPO, équipe data, RSSI selon contexte).

2. **Droit de recours ?**  
   Oui. Les personnes doivent pouvoir contester les décisions automatisées/profilages et obtenir intervention humaine.

3. **Organisme de contrôle**  
   En France : **CNIL** (protection des données). Selon secteur, d'autres autorités peuvent intervenir.


## Slide synthèse (template TP2)

**CAS : A (Churn clients)**  
**MODÈLE RETENU : Logistic Regression**

**MÉTRIQUES FINALES**  
Accuracy : **80.38%**  
F1-macro : **0.739**  
F1-weighted : **0.800**

**TOP 3 VARIABLES (SHAP)**  
1. `tenure`  
2. `InternetService_Fiber optic`  
3. `Contract_Two year`

**AI ACT** : Niveau **Risque limité** — système d'aide à la décision commerciale avec impact modéré, sous supervision humaine.

**LIMITE PRINCIPALE IDENTIFIÉE** : rappel de la classe churn encore perfectible (faux négatifs à réduire via calibration de seuil et stratégie coût-sensible).
